In [ ]:
#!/usr/bin/env python3

import numpy as np
import pandas as pd
import time,os,sys
import numpy as np
import csv 
import pandas as pd
from scipy.optimize import minimize, brentq
import progressbar as P

sys.path.append('../Recast/CMS-TOP-20-001_mtt')

from cms_top_20_001_Limits import read_CMSdata, getSMLO, getKfactor, cms_bins, bin_widths, getUL

In [31]:
def getBinsFrom(inputFile="../pp2ttbar-loop_smefit/Events/run_01/MADatNLO.HwU",
                hist_label="CMS_2021_mtt-",weightFactor=2*0.67*0.2):
    
    with open(inputFile,'r') as f:
        data = f.read().split('<histogram>')[1:]

    histogram = []
    for hist in data:
        lines = hist.split('\n')
        if not (hist_label in lines[0]):
            continue
        low_edge,high_edge,bin_value = lines[1].split()[:3]
        histogram.append([float(low_edge),float(high_edge),float(bin_value)*weightFactor ])
    
    return histogram


In [32]:
def computeULs(histogram,full=False):

    # ### Load CMS data and BG
    xsecsObs,sm,covMatrix = read_CMSdata(dataDir='../Recast/CMS-TOP-20-001_mtt/data')
    
    # ### Load LO background from MG5
    smLO = getSMLO(smFile='../Recast/CMS-TOP-20-001_mtt/sm/sm_tt_lo_cms_top_20_001.pcl')
    # Get k-factor for each bin
    kfac = getKfactor(sm,smLO)

    
    bins_left = [pt[0] for pt in histogram]
    bins_right = [pt[1] for pt in histogram]
    bin_values = [pt[2] for pt in histogram]
    # Check that bins are consistent:
    if not np.array_equal(bins_left,cms_bins[:-1]):
        print('Bins from data do not match CMS')
        return
    if bins_right[-1] != cms_bins[-1]:
        print('Bins from data do not match CMS')
        return

    signal = list(zip(bins_left,bin_values))
    signal = np.array(sorted(signal))[:,1]
    yDM = 1.0
    # Make sure signal is normalized to yDM = 1
    signal = signal/yDM**2
    # Rescale predictions by bin-dependent k-factors
    signal = kfac*signal

    if not full: # Use simplified chi-square
        # Finally, divide by the bin widths
        signal = signal/bin_widths
        sm_bin = sm/bin_widths
        resDict = getUL(signal,sm_bin,xsecsObs,covMatrix,deltas=0.0)
        yDM95 = resDict['yDM95']
        deltaChi95 = resDict['deltaChi95']       

        # Expected
        resDictExp = getUL(signal,sm_bin,sm_bin,covMatrix,deltas=0.0)
        yDM95exp = resDictExp['yDM95']            
        
    else: # Use full CLs calculation
        import sys
        sys.path.append('../statisticalTools')
        from simplifiedLikelihoods import Data,UpperLimitComputer,LikelihoodComputer
        ulComp = UpperLimitComputer()

        # ### Get number of observed and expected (BG) events
        lumi = 137*1e3
        nobs = xsecsObs*bin_widths*lumi
        nbg = sm*lumi
        ns = signal*lumi
        cov = covMatrix*(lumi*bin_widths)**2
        data = Data(observed=nobs, backgrounds=nbg, 
                    covariance=cov, 
                    nsignal=ns,deltas_rel=0.0)
        ul = ulComp.getUpperLimitOnMu(data)
        yDM95 = np.sqrt(ul)
        # Signal for 95% C.L. limit:
        data95 = Data(observed=nobs, backgrounds=nbg, 
                        covariance=cov, 
                        nsignal=ns*ul,deltas_rel=0.0)
        computer = LikelihoodComputer(data95)
        deltaChi95 = computer.chi2()

        yDM95exp = None

    return yDM95,yDM95exp,deltaChi95

In [35]:
# Since the tops have not decayed and the CMS data refers to semi-leptonic ttbar, we must multiply by
# BR(W+ -> l+ nu)*BR(W- -> hadronic) + BR(W- -> l- nu)*BR(W+ -> hadronic) = 2*0.2*0.67
weightFactot = 2*0.67*0.2 
histogram = getBinsFrom("../pp2ttbar-loop_smefit/Events/run_01/MADatNLO.HwU","CMS_2021_mtt-",weightFactot)
yDM95,yDM95exp,deltaChi95 = computeULs(histogram,full=False)
print(f"yDM (obs) @ 95% C.L. = {yDM95:1.2f}")
print(f"yDM (exp) @ 95% C.L. = {yDM95exp:1.2f}")

yDM (obs) @ 95% C.L. = 4.93
yDM (exp) @ 95% C.L. = 8.49
